# Need Detection Retrain — XLM-RoBERTa, 5 languages, multi-label
### Capstone 160 -- Favaz (Task 5)

Retrains Need Detection on the **expanded** multilingual dataset (3,000 rows/language, up from 1,500 in Task 4), and fixes two real methodology bugs found in Task 4:

1. **Leakage-safe split.** Task 4 split randomly across the combined table, so a tweet's English version could land in training while its Kannada translation landed in test -- the model could effectively "already know" a test tweet from its English twin, inflating per-language scores. This notebook splits by the tweet's original identity (`tweet_id`) first, so all language versions of a given tweet stay together in the same split -- same fix used in the disaster classifier and language detection notebooks.
2. **Threshold tuning on a held-out validation set, not the test set.** Task 4 tuned per-label thresholds on the same test set it then reported as final numbers -- overfits to that specific sample. Fixed here: thresholds are chosen on `val_df`, only applied to `test_df`.

Also fixes the same root cause as everywhere else in this project: **language imbalance**. Task 4's Kannada macro-F1 was 0.467 (vs Hindi 0.730, Tamil 0.715, Telugu 0.668) with no correction applied -- non-English training rows are oversampled here, same fix that worked for the disaster classifier.

Keeps what was already correct in Task 2/4: multi-label sigmoid output (a tweet can hit 0, 1, or several departments), `BCEWithLogitsLoss` with per-label `pos_weight` for the label imbalance (`is_tourism` is rare), and `xlm-roberta-base`.

**Before running anything, set up your Kaggle notebook:**
1. Right sidebar -> **Settings -> Accelerator -> GPU T4 x2** (or P100).
2. Right sidebar -> **Settings -> Internet -> On**.
3. Right sidebar -> **Add Input -> Upload -> Upload Dataset**, and upload:
   - `disaster_tweets_preprocessed_v2.csv` -- full 11,095-row English dataset (has `is_healthcare`/`is_municipal`/`is_tourism`/`is_law`)
   - `disaster_tweets_multilingual_v2.csv` -- the 3,000-row expanded translation file (also carries those same 4 label columns through)

**Output**: per-tweet `needs` dict (healthcare/municipal/tourism/law scores 0-1) and `need_confidence` (max score) -- matching the locked pipeline schema. This is the second of your two Step 3a/3b-parallel outputs to build (the other being Akshata's Sentiment Analysis).


In [49]:
!pip uninstall -y torchvision torchaudio -q
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.46.3" datasets scikit-learn accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [50]:
import pandas as pd
import numpy as np
import torch
import glob
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected -- check Settings -> Accelerator, then restart the session.")

Using device: cuda


## Step 1 -- Load and combine English + all 4 translated languages

Same long-format approach as the disaster classifier: one row per (tweet, language) pair, with a stable `tweet_id` shared across all language versions of the same original tweet. Each translated row reuses the English original's department labels -- translating a tweet doesn't change which department it's about.


In [51]:
def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under /kaggle/input/. "
            "Make sure you added it via Add Input -> Upload -> Upload Dataset."
        )
    return matches[0]

LABEL_COLS = ['is_healthcare', 'is_municipal', 'is_tourism', 'is_law']
LANG_COLS = {'clean_tweet_hindi': 'hi', 'clean_tweet_kannada': 'kn', 'clean_tweet_tamil': 'ta', 'clean_tweet_telugu': 'te'}

df_full = pd.read_csv(find_file("disaster_tweets_preprocessed_v2 (1).csv"))
df_multi = pd.read_csv(find_file("disaster_tweets_multilingual_v2.csv"))

for name, df, cols in [("preprocessed", df_full, ['tweet', 'clean_tweet'] + LABEL_COLS),
                        ("multilingual", df_multi, ['tweet'] + list(LANG_COLS.keys()) + LABEL_COLS)]:
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f"{name} file missing columns: {missing}. Found: {df.columns.tolist()}"

df_full = df_full.reset_index(drop=True)
df_full['tweet_id'] = [f"tw_{i:05d}" for i in range(len(df_full))]
tweet_to_id = dict(zip(df_full['tweet'].astype(str), df_full['tweet_id']))

unmatched = (~df_multi['tweet'].astype(str).isin(tweet_to_id)).sum()
assert unmatched == 0, f"{unmatched} rows in the multilingual file don't match any tweet in the full dataset."

rows = []
for _, r in df_full.iterrows():
    rows.append({'tweet_id': r['tweet_id'], 'text': r['clean_tweet'], 'lang': 'en',
                 **{c: int(r[c]) for c in LABEL_COLS}})
for _, r in df_multi.iterrows():
    tid = tweet_to_id[str(r['tweet'])]
    labels = {c: int(r[c]) for c in LABEL_COLS}
    for col, code_ in LANG_COLS.items():
        text = r[col]
        if pd.isna(text) or str(text).strip() == "":
            continue
        rows.append({'tweet_id': tid, 'text': text, 'lang': code_, **labels})

combined_df = pd.DataFrame(rows)
print("Combined shape:", combined_df.shape)
print("Per-language counts:\n", combined_df['lang'].value_counts())
print("\nLabel sums (combined):\n", combined_df[LABEL_COLS].sum())

Combined shape: (23095, 7)
Per-language counts:
 lang
en    11095
hi     3000
kn     3000
ta     3000
te     3000
Name: count, dtype: int64

Label sums (combined):
 is_healthcare     629
is_municipal     1783
is_tourism        143
is_law            584
dtype: int64


## Step 2 -- Leakage-safe train/val/test split (80/10/10)

Splits by the tweet's original identity, not by row -- so all 5 language versions of a given tweet always land in the same split. Uses a coarse multi-label stratification key (exact label combination when it has enough examples, otherwise a "rare" fallback bucket, same approach Task 2/4 used) so the split stays balanced across department labels without erroring out on rare combos like `is_tourism`.


In [52]:
from sklearn.model_selection import train_test_split

def make_strat_key(row):
    return tuple(row[LABEL_COLS])

unique_tweets = df_full[['tweet_id'] + LABEL_COLS].drop_duplicates(subset=['tweet_id']).copy()
unique_tweets['_key'] = unique_tweets[LABEL_COLS].apply(make_strat_key, axis=1)
combo_counts = unique_tweets['_key'].value_counts()
rare_combos = combo_counts[combo_counts < 10].index
unique_tweets['_key_safe'] = unique_tweets['_key'].apply(lambda k: str(k) if k not in rare_combos else 'rare')

train_ids, temp_ids = train_test_split(unique_tweets, test_size=0.2, random_state=42, stratify=unique_tweets['_key_safe'])
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42, stratify=temp_ids['_key_safe'])

train_df = combined_df[combined_df['tweet_id'].isin(train_ids['tweet_id'])].reset_index(drop=True)
val_df   = combined_df[combined_df['tweet_id'].isin(val_ids['tweet_id'])].reset_index(drop=True)
test_df  = combined_df[combined_df['tweet_id'].isin(test_ids['tweet_id'])].reset_index(drop=True)

print("Train / val / test row counts:", train_df.shape[0], val_df.shape[0], test_df.shape[0])

overlap = (set(train_df['tweet_id']) & set(val_df['tweet_id'])) | \
          (set(train_df['tweet_id']) & set(test_df['tweet_id'])) | \
          (set(val_df['tweet_id']) & set(test_df['tweet_id']))
assert not overlap, f"LEAKAGE DETECTED: {overlap}"
print("No leakage between splits, confirmed.")

print("\nTrain language counts:\n", train_df['lang'].value_counts())
print("\nTest label sums:\n", test_df[LABEL_COLS].sum())

Train / val / test row counts: 18424 2329 2342
No leakage between splits, confirmed.

Train language counts:
 lang
en    8876
hi    2387
kn    2387
ta    2387
te    2387
Name: count, dtype: int64

Test label sums:
 is_healthcare     71
is_municipal     169
is_tourism        15
is_law            58
dtype: int64


## Step 3 -- Balance language representation in training

Same problem as the disaster classifier: English outnumbers each other language roughly 3.7:1 in training. Oversampling non-English training rows 3x brings each language closer to equal representation -- `val_df` and `test_df` are left completely untouched, so evaluation stays honest.


In [53]:
OVERSAMPLE_FACTOR = 3

print("Before oversampling:")
print(train_df['lang'].value_counts())

non_en_train = train_df[train_df['lang'] != 'en']
oversampled = pd.concat([non_en_train] * (OVERSAMPLE_FACTOR - 1), ignore_index=True)
train_df = pd.concat([train_df, oversampled], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter oversampling:")
print(train_df['lang'].value_counts())
print("\nNew train_df size:", train_df.shape[0], "(val_df and test_df unchanged)")

Before oversampling:
lang
en    8876
hi    2387
kn    2387
ta    2387
te    2387
Name: count, dtype: int64

After oversampling:
lang
en    8876
ta    7161
kn    7161
te    7161
hi    7161
Name: count, dtype: int64

New train_df size: 37520 (val_df and test_df unchanged)


In [54]:
AUGMENT_EXAMPLES = [
    ("the ground shook so hard buildings came down", {'is_municipal': 1}),
    ("the streets are underwater and homes are collapsing", {'is_municipal': 1}),
    ("thick black smoke pouring out of the factory roof", {'is_municipal': 1}),
    ("the bridge just gave way with cars still on it", {'is_municipal': 1}),
    ("whole neighborhood without power after the storm hit", {'is_municipal': 1}),
    ("a huge chunk of the hillside slid onto the road", {'is_municipal': 1}),
    ("the dam is leaking and villages downstream are being told to leave", {'is_municipal': 1}),
    ("windows shattered from the blast two streets over", {'is_municipal': 1}),
    ("the tremor knocked out the water supply for the whole town", {'is_municipal': 1}),
    ("smoke and sirens everywhere near the old market", {'is_municipal': 1}),
    ("gangs are looting shops and nobody is stopping them", {'is_law': 1}),
    ("people were dragged out of their homes and beaten", {'is_law': 1}),
    ("mob smashing car windows down the main road", {'is_law': 1}),
    ("someone pulled a knife on the bus and ran off", {'is_law': 1}),
    ("police fired tear gas to break up the crowd", {'is_law': 1}),
    ("masked men broke into the store and grabbed everything", {'is_law': 1}),
    ("protesters clashed with officers outside the building", {'is_law': 1}),
    ("he was chased down the street and robbed at knifepoint", {'is_law': 1}),
    ("shots fired near the school, everyone told to stay inside", {'is_law': 1}),
    ("group of men vandalizing storefronts downtown", {'is_law': 1}),
    ("dozens injured, ambulances can't get through the crowd", {'is_healthcare': 1}),
    ("she needs urgent medical help and there's no doctor nearby", {'is_healthcare': 1}),
    ("stranded at the resort with no flights out for days", {'is_tourism': 1}),
    ("tourists trapped at the hotel after roads were closed", {'is_tourism': 1}),
    ("backpackers stuck at the border after the crossing shut down", {'is_tourism': 1}),
    ("the cruise ship was forced to dock early after the storm warning", {'is_tourism': 1}),
    ("hikers went missing on the trail near the national park", {'is_tourism': 1}),
    ("the beach resort evacuated all guests before the cyclone hit", {'is_tourism': 1}),
    ("our tour group got stranded when the volcano erupted nearby", {'is_tourism': 1}),
    ("the airport shut down leaving hundreds of travelers stranded", {'is_tourism': 1}),
    ("visitors were told to leave the island before the hurricane made landfall", {'is_tourism': 1}),
    ("the safari lodge had to evacuate tourists after the wildfire spread", {'is_tourism': 1}),
]

AUG_REPEAT = 5
aug_rows = []
for text, labels in AUGMENT_EXAMPLES:
    row = {'tweet_id': f'aug_{len(aug_rows):04d}', 'text': text, 'lang': 'en',
           'is_healthcare': 0, 'is_municipal': 0, 'is_tourism': 0, 'is_law': 0}
    row.update(labels)
    aug_rows.append(row)

aug_df = pd.DataFrame(aug_rows * AUG_REPEAT)
train_df = pd.concat([train_df, aug_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print("Train size after augmentation:", train_df.shape[0])
print(train_df[LABEL_COLS].sum())

Train size after augmentation: 37680
is_healthcare     999
is_municipal     2986
is_tourism        250
is_law           1009
dtype: int64


In [55]:
tourism_rows = train_df[train_df['is_tourism'] == 1]
train_df = pd.concat([train_df, tourism_rows, tourism_rows], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print("Train size after tourism boost:", train_df.shape[0])
print("is_tourism positives now:", train_df['is_tourism'].sum())

Train size after tourism boost: 38180
is_tourism positives now: 750


## Step 4 -- Tokenize


In [56]:
from transformers import XLMRobertaTokenizerFast
from datasets import Dataset

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128

tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_df(df):
    ds = Dataset.from_pandas(df[['text'] + LABEL_COLS].astype({'text': str}))
    def _tok(batch):
        enc = tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH)
        enc['labels'] = [[float(batch[c][i]) for c in LABEL_COLS] for i in range(len(batch['text']))]
        return enc
    ds = ds.map(_tok, batched=True)
    ds = ds.remove_columns(['text'] + LABEL_COLS)
    ds.set_format('torch')
    return ds

train_ds = tokenize_df(train_df)
val_ds   = tokenize_df(val_df)
test_ds  = tokenize_df(test_df)

print("Tokenized dataset sizes:", len(train_ds), len(val_ds), len(test_ds))

Map:   0%|          | 0/38180 [00:00<?, ? examples/s]

Map:   0%|          | 0/2329 [00:00<?, ? examples/s]

Map:   0%|          | 0/2342 [00:00<?, ? examples/s]

Tokenized dataset sizes: 38180 2329 2342


In [57]:
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

## Step 5 -- Model with class-imbalance handling (pos_weight per label)

`is_tourism` is rare, so plain BCE loss would barely notice it. `pos_weight` (per-label negative:positive ratio, capped at 10 to avoid instability) fixes that -- same approach as Task 2/4, recomputed here on the oversampled training set.


In [58]:
from torch import nn
from transformers import Trainer, TrainingArguments, XLMRobertaForSequenceClassification

train_labels_arr = train_df[LABEL_COLS].values.astype(np.float32)
pos_counts = train_labels_arr.sum(axis=0)
neg_counts = len(train_labels_arr) - pos_counts
pos_weight = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 10.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float).to(device)

print("Labels:    ", LABEL_COLS)
print("Pos counts:", pos_counts)
print("Pos weight:", pos_weight.cpu().numpy())

class WeightedMultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

model = XLMRobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABEL_COLS), problem_type="multi_label_classification"
).to(device)

Labels:     ['is_healthcare', 'is_municipal', 'is_tourism', 'is_law']
Pos counts: [1029. 2990.  750. 1009.]
Pos weight: [10. 10. 10. 10.]


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step 6 -- Train

Larger training set than Task 4 (~37,500 rows after oversampling, vs ~13,700) -- 3 epochs given the time budget; raise it if you have time to spare.


In [59]:
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
    return {'f1_macro': f1_macro}

training_args = TrainingArguments(
    output_dir="/kaggle/working/need_detection_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none",
    seed=42,
)

trainer = WeightedMultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.304100,0.227248,0.558013
2,0.111300,0.211392,0.758586
3,0.036500,0.240171,0.756084


TrainOutput(global_step=3582, training_loss=0.2384478880133062, metrics={'train_runtime': 1490.5515, 'train_samples_per_second': 76.844, 'train_steps_per_second': 2.403, 'total_flos': 7534320363048960.0, 'train_loss': 0.2384478880133062, 'epoch': 3.0})

## Step 7 -- Per-label threshold tuning (on val_df -- fixes Task 4's bug)

Task 4 tuned thresholds on the same test set it reported as final numbers. This tunes on `val_df` instead, then only applies the resulting thresholds to `test_df` in Step 8.


In [60]:
def get_probs(dataset):
    preds = trainer.predict(dataset)
    logits = preds.predictions
    probs = 1 / (1 + np.exp(-logits))
    return probs

val_probs = get_probs(val_ds)
val_labels = val_df[LABEL_COLS].values.astype(int)

best_thresholds = {}
print("Per-label threshold search (on val_df):\n")
for i, label_name in enumerate(LABEL_COLS):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.2, 0.81, 0.05):
        preds = (val_probs[:, i] >= t).astype(int)
        f1 = f1_score(val_labels[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    best_thresholds[label_name] = float(best_t)
    preds = (val_probs[:, i] >= best_t).astype(int)
    p = precision_score(val_labels[:, i], preds, zero_division=0)
    r = recall_score(val_labels[:, i], preds, zero_division=0)
    print(f"{label_name:15s}  threshold={best_t:.2f}  val_precision={p:.2f}  val_recall={r:.2f}  val_f1={best_f1:.2f}")

print("\nFinal per-label thresholds (chosen on val_df):", best_thresholds)

Per-label threshold search (on val_df):

is_healthcare    threshold=0.65  val_precision=0.88  val_recall=0.86  val_f1=0.87
is_municipal     threshold=0.80  val_precision=0.74  val_recall=0.88  val_f1=0.80
is_tourism       threshold=0.75  val_precision=0.88  val_recall=0.62  val_f1=0.73
is_law           threshold=0.65  val_precision=0.57  val_recall=0.91  val_f1=0.70

Final per-label thresholds (chosen on val_df): {'is_healthcare': 0.6499999999999999, 'is_municipal': 0.7999999999999998, 'is_tourism': 0.7499999999999998, 'is_law': 0.6499999999999999}


## Step 8 -- Evaluate on the held-out test set: overall, then per-language (the important one)

Per Task 4's own reasoning (kept here because it was correct): the per-language breakdown is the real check. Since Hindi/Kannada/Tamil/Telugu training rows are translations, a translated tweet doesn't contain the original English keyword its label came from -- the model has to actually understand the translated meaning to get it right. Good, *balanced* per-language scores here are real evidence of generalization.


In [61]:
test_probs = get_probs(test_ds)
test_labels = test_df[LABEL_COLS].values.astype(int)
test_langs = test_df['lang'].values

final_preds = np.zeros_like(test_probs)
for i, label_name in enumerate(LABEL_COLS):
    final_preds[:, i] = (test_probs[:, i] >= best_thresholds[label_name]).astype(int)

from sklearn.metrics import classification_report
print("=" * 60)
print("OVERALL RESULTS (all languages combined, thresholds from val_df)")
print("=" * 60)
print(classification_report(test_labels, final_preds, target_names=LABEL_COLS, zero_division=0))
print(f"Hamming loss (lower is better): {hamming_loss(test_labels, final_preds):.4f}")
print(f"Exact-match (subset) accuracy:  {(final_preds == test_labels).all(axis=1).mean()*100:.2f}%")

print("\n" + "=" * 60)
print("PER-LANGUAGE BREAKDOWN")
print("=" * 60)
print(f"{'Language':10s} {'N':>6s} {'Exact-match %':>15s} {'Macro F1':>10s}")
for lang in sorted(np.unique(test_langs)):
    mask = test_langs == lang
    labels_l = test_labels[mask]
    preds_l = final_preds[mask]
    exact_match = (preds_l == labels_l).all(axis=1).mean() * 100 if mask.sum() else float('nan')
    f1_macro = f1_score(labels_l, preds_l, average='macro', zero_division=0) if mask.sum() else float('nan')
    print(f"{lang:10s} {mask.sum():6d} {exact_match:15.2f} {f1_macro:10.3f}")

OVERALL RESULTS (all languages combined, thresholds from val_df)
               precision    recall  f1-score   support

is_healthcare       0.71      0.72      0.71        71
 is_municipal       0.69      0.79      0.74       169
   is_tourism       0.38      0.40      0.39        15
       is_law       0.76      0.91      0.83        58

    micro avg       0.70      0.78      0.73       313
    macro avg       0.63      0.71      0.67       313
 weighted avg       0.69      0.78      0.73       313
  samples avg       0.10      0.10      0.10       313

Hamming loss (lower is better): 0.0188
Exact-match (subset) accuracy:  92.66%

PER-LANGUAGE BREAKDOWN
Language        N   Exact-match %   Macro F1
en           1110           98.65      0.919
hi            308           90.58      0.553
kn            308           87.01      0.376
ta            308           87.01      0.421
te            308           84.42      0.415


## Step 9 -- Sanity check on hand-written, keyword-free examples

Same style as Task 2/4's stress tests -- sentences describing real needs without the literal keywords used to build the original labels, in multiple languages this time.


In [62]:
model.eval()

def predict_needs(text):
    enc = tokenizer(text, max_length=MAX_LENGTH, padding='max_length', truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(**enc).logits).cpu().numpy()[0]
    return {name: float(probs[i]) for i, name in enumerate(LABEL_COLS)}

sanity_tests = [
    ("massive earthquake hits coastal town, hospitals overwhelmed", "en", "is_healthcare"),
    ("gangs are looting shops and nobody is stopping them", "en", "is_law"),
    ("बाढ़ आ गई है, मदद चाहिए", "hi", "is_municipal"),
    ("ಪ್ರವಾಹ ಬಂದಿದೆ ಸಹಾಯ ಬೇಕು", "kn", "is_municipal"),
    ("just had lunch with my family, lovely day", "en", None),
]

for text, lang, expected in sanity_tests:
    probs = predict_needs(text)
    predicted = [k for k in LABEL_COLS if probs[k] >= best_thresholds[k]]
    match = (expected in predicted) if expected else (len(predicted) == 0)
    print(f"[{'yes' if match else 'NO '}] [{lang}] {text!r:55s} predicted={predicted} expected={expected}")
    print(f"       probs: {{{', '.join(f'{k}: {v:.3f}' for k, v in probs.items())}}}")

[NO ] [en] 'massive earthquake hits coastal town, hospitals overwhelmed' predicted=['is_municipal'] expected=is_healthcare
       probs: {is_healthcare: 0.139, is_municipal: 0.998, is_tourism: 0.025, is_law: 0.026}
[yes] [en] 'gangs are looting shops and nobody is stopping them'   predicted=['is_law'] expected=is_law
       probs: {is_healthcare: 0.042, is_municipal: 0.121, is_tourism: 0.080, is_law: 0.997}
[NO ] [hi] 'बाढ़ आ गई है, मदद चाहिए'                               predicted=[] expected=is_municipal
       probs: {is_healthcare: 0.071, is_municipal: 0.260, is_tourism: 0.003, is_law: 0.004}
[NO ] [kn] 'ಪ್ರವಾಹ ಬಂದಿದೆ ಸಹಾಯ ಬೇಕು'                               predicted=[] expected=is_municipal
       probs: {is_healthcare: 0.004, is_municipal: 0.017, is_tourism: 0.002, is_law: 0.004}
[yes] [en] 'just had lunch with my family, lovely day'             predicted=[] expected=None
       probs: {is_healthcare: 0.005, is_municipal: 0.006, is_tourism: 0.008, is_law: 0.004}


## Step 10 -- Generate final predictions and save the model


In [63]:
output_records = []
for i in range(len(test_df)):
    row = test_df.iloc[i]
    needs = {name.replace('is_', ''): round(float(test_probs[i, j]), 4) for j, name in enumerate(LABEL_COLS)}
    output_records.append({
        "tweet_id": row['tweet_id'],
        "text": row['text'],
        "lang": row['lang'],
        "needs": needs,
        "need_confidence": round(float(max(needs.values())), 4),
    })

import json
with open("/kaggle/working/need_detection_predictions.json", "w", encoding="utf-8") as f:
    json.dump(output_records, f, ensure_ascii=False, indent=2)
print("Saved /kaggle/working/need_detection_predictions.json")
print("Sample:", output_records[0])

model.save_pretrained("/kaggle/working/need_detection_xlmr")
tokenizer.save_pretrained("/kaggle/working/need_detection_xlmr")
with open("/kaggle/working/need_detection_xlmr/thresholds.json", "w") as f:
    json.dump(best_thresholds, f, indent=2)

import shutil
shutil.make_archive("/kaggle/working/need_detection_xlmr", "zip", "/kaggle/working/need_detection_xlmr")
print("\nSaved model + tokenizer + thresholds to /kaggle/working/need_detection_xlmr")
print("Zipped to /kaggle/working/need_detection_xlmr.zip -- download the zip (individual large files can fail to download on Kaggle).")

Saved /kaggle/working/need_detection_predictions.json
Sample: {'tweet_id': 'tw_00003', 'text': 'arsonist sets cars ablaze at dealership', 'lang': 'en', 'needs': {'healthcare': 0.036, 'municipal': 0.9984, 'tourism': 0.0288, 'law': 0.0394}, 'need_confidence': 0.9984}

Saved model + tokenizer + thresholds to /kaggle/working/need_detection_xlmr
Zipped to /kaggle/working/need_detection_xlmr.zip -- download the zip (individual large files can fail to download on Kaggle).


## What's next

1. Check the Step 8 per-language table first -- if Kannada is now close to the other languages (unlike Task 4's 0.467 macro-F1), the fixes worked.
2. This model + `predict_needs()` is one of the two Step 3a/3b-parallel outputs -- the other is Akshata's Sentiment Analysis, also consuming the post-language-detection tweet independently.
3. Once both are done, the merge step combines `needs`/`need_confidence` from this model with the sentiment fields into the final locked schema.
